In [ ]:
import torch
import torch.nn as nn 
import torch.optim as optim 
from torchvision import datasets, transforms 
from torch.utils.data import DataLoader

class TorchSimpleConvNet(nn.Module):
    def __init__(self):
        super(TorchSimpleConvNet,self).__init__()
        self.conv1=nn.Conv2d(in_channels=1,out_channels=30,kernel_size=5,stride=1,padding=0)
        self.pool=nn.MaxPool2d(kernel_size=2,stride=2)

        self.fc1=nn.Linear(in_features=4320,out_features=100)
        self.fc2=nn.Linear(in_features=100,out_features=10)
        self.relu=nn.ReLU()

    def forward(self,x):
        x=self.conv1(x)
        x=self.relu(x)
        x=self.pool(x)
        x=x.view(x.size(0),-1)

        x=self.fc1(x)
        x=self.relu(x)
        x=self.fc2(x)

        return x


In [2]:
# 1. 데이터셋 및 데이터로더 준비 (텐서 변환 및 정규화 자동 수행)
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# 배치 단위로 데이터를 묶어주는 DataLoader
train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False)

# 2. 모델, 오차 함수, 옵티마이저 생성
model = TorchSimpleConvNet()

# Softmax + Cross Entropy가 합쳐진 함수
criterion = nn.CrossEntropyLoss() 

# 가중치 업데이트를 담당할 옵티마이저 (학습률 0.1의 SGD)
optimizer = optim.SGD(model.parameters(), lr=0.1)

# 3. 학습 루프
epochs = 2 # 전체 데이터를 반복할 횟수 (1 에포크 = 미니배치 여러 번 돌아 전체 데이터 1번 다 봄)

print("=== PyTorch 학습 시작 ===")
for epoch in range(epochs):
    model.train() # 모델을 학습 모드로 설정
    
    for i, (images, labels) in enumerate(train_loader):
        # 1. 기울기 초기화 (이전 배치의 기울기가 남아있는 것을 방지)
        optimizer.zero_grad()
        
        # 2. 순전파
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 3. 역전파 (마법의 한 줄! 모든 backward 계산과 col2im을 대신 수행)
        loss.backward()
        
        # 4. 가중치 업데이트 (파라미터 갱신)
        optimizer.step()
        
        if (i+1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

print("=== 학습 완료 ===")

100%|██████████| 9.91M/9.91M [00:02<00:00, 3.47MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 143kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.35MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.32MB/s]


NameError: name 'TorchSimpleConvNet' is not defined

In [ ]:
model.eval() # 모델을 평가 모드로 설정 (불필요한 연산 방지)
correct = 0
total = 0

# 역전파(기울기 계산)를 하지 않도록 설정하여 메모리 절약 및 속도 향상
with torch.no_grad(): 
    for images, labels in test_loader:
        outputs = model(images)
        
        # 가장 높은 값을 가진 인덱스를 예측값으로 추출
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"최종 테스트 정확도 (Accuracy): {100 * correct / total:.2f}%")